# graph-imgrag — Experiment Notebook
Run the full pipeline interactively.

In [ ]:
import sys
sys.path.insert(0, '..')
from src.utils.helpers import load_config
cfg = load_config('../configs/config.yaml')
print('Config loaded:', cfg['dataset']['num_images'], 'images')

In [ ]:
# Create demo dataset
import sys; sys.path.insert(0, '..')
from main import create_demo_dataset
create_demo_dataset(cfg)

In [ ]:
# OCR
from src.ocr.extract_text import run_ocr
ocr_results = run_ocr(cfg)
print(f'OCR done: {len(ocr_results)} images')

In [ ]:
# Embeddings
from src.embeddings.generate_embeddings import generate
image_paths, embeddings = generate(ocr_results, cfg)
labels = [os.path.basename(p) for p in image_paths]
print(f'Embeddings: {embeddings.shape}')

In [ ]:
# Graph
import os
from src.graph.build_graph import build, visualize
G, sim_matrix, labels = build(image_paths, embeddings, cfg)
visualize(G, labels, cfg)
print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

In [ ]:
# Search
from src.retrieval.search import search
results = search('passport', image_paths, embeddings, G, labels, cfg)
for r in results:
    print(f"#{r['rank']}  {r['file']}  sim={r['similarity']:.4f}")

In [ ]:
# Evaluate
from src.retrieval.search import evaluate
metrics = evaluate(image_paths, embeddings, G, labels, cfg)
print('Mean Avg Precision:', metrics['mean_avg_precision'])